In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

In [ ]:
print("\n--- 1. VERİ HAZIRLIĞI ---")

data = {
    'text': [
        "Hesap bakiyemi ve son üç işlemimi öğrenmek istiyorum.", # Güvenli
        "Kredi kartı borç ödeme tarihim ne zaman?", # Güvenli
        "ACİL: Hesabınızdan 10.000 TL çekildi. Durdurmak için: bit.ly/banka-onay", # Fraud
        "Tebrikler, çekilişle iPhone kazandınız! Bilgi için hemen arayın.", # Fraud
        "Adres değişikliği talebim neden hala onaylanmadı?", # Güvenli
        "Giriş şifreniz kilitlendi. Kilidi açmak için bu mesaja 'EVET' yazın.", # Fraud
        "Kredi başvurumun sonuçlanmasını bekliyorum, teşekkürler.", # Güvenli
        "SAYIN MUSTERI, kart aidat iadenizi hemen almak için tıklayın: iade-bank.com", # Fraud
        "Yurt dışı tatili için döviz hesabı açtırmak istiyorum.", # Güvenli
        "Hesabınızda şüpheli hareket görüldü. Güvenlik için şifrenizi SMS ile gönderin.", # Fraud
        "Yeni kimlik kartım ile banka hesabı açabilir miyim?", # Güvenli
        "Müşteri hizmetlerine bağlanamıyorum, acil destek lütfen.", # Güvenli
        "BBDK onayıyla hesabınızdaki bloke kalktı. Bilgilerinizi güncellemek için tıklayın: guncel-bank.org", # Fraud
        "Acil para transferi yapmam gerekiyor, limitim yeterli mi?", # Güvenli
        "Merhaba, IBAN numaramı öğrenebilir miyim?" # Güvenli
    ],
    'is_fraud': [0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0] # 0: Güvenli, 1: Fraud
}

df = pd.DataFrame(data)

# Veri Görselleştirme: Sınıf Dağılımı
plt.figure(figsize=(7, 5))
sns.countplot(x='is_fraud', hue='is_fraud',data=df, palette='viridis')
plt.title('Veri Seti Sınıf Dağılımı (0: Güvenli, 1: Fraud)')
plt.xlabel('Sınıf')
plt.ylabel('Mesaj Sayısı')
plt.xticks(ticks=[0, 1], labels=['Güvenli', 'Dolandırıcılık'])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print(f"Toplam Mesaj Sayısı: {len(df)}")
print(f"Güvenli Mesaj Sayısı: {df['is_fraud'].value_counts()[0]}")
print(f"Dolandırıcılık Mesaj Sayısı: {df['is_fraud'].value_counts()[1]}")

In [ ]:
print("\n--- 2. MODEL EĞİTİMİ VE KELİME ÖNEM ANALİZİ ---")

# Veriyi Eğitim ve Test Setlerine Ayırma (Küçük veri setinde train_test_split simülatif olarak kullanılıyor)
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['is_fraud'], test_size=0.3, random_state=42, stratify=df['is_fraud'])

# Metin Vektörizasyonu (TF-IDF)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=100) # Bağlam için 1 ve 2'li kelime grupları
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Model Eğitimi (Random Forest Classifier)
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced') # Sınıf dengesizliği için 'balanced'
model.fit(X_train_vec, y_train)

# Modelin Karar Verirken En Çok Odaklandığı Kelimeler (Feature Importance)
features = vectorizer.get_feature_names_out()
importances = model.feature_importances_
indices = np.argsort(importances)[-15:] # En önemli 15 kelime/kelime grubunu al

plt.figure(figsize=(10, 7))
plt.title('Modelin Karar Verirken En Çok Odaklandığı Kelimeler/Gruplar')
sns.barplot(x=importances[indices],y=[features[i] for i in indices],hue=[features[i] for i in indices],palette='coolwarm',legend=False)
plt.xlabel('Göreceli Önem Düzeyi (Feature Importance)')
plt.ylabel('Kelime/Kelime Grubu')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
print("\n--- 3. AI TESTING: EDGE CASE SENARYOLARI ---")

def ai_predict_with_confidence(sample_text):
 #Verilen metin için tahmin yapar ve güven skoru ile birlikte döner.
    vec = vectorizer.transform([sample_text])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0][1] # Fraud olma olasılığı
    return 'FRAUD' if pred == 1 else 'GÜVENLİ', prob

print("----------------------------------------------------------------")

# 3.1. Adversarial Text Manipulation (Yanıltıcı Metin Manipülasyonu)
print("\n3.1. Adversarial Text Manipulation (Yanıltıcı Metin Manipülasyonu)")
manipulated_texts = [
    "H.e.s.a.p i.a.d.e.s.i için tıkla",       # Harf arası nokta
    "KRED1 KART1 B0RCU",                    # Rakamla harf değiştirme (Leetspeak)
    "P a r a k a z a n d ı n ı z",          # Harf arası boşluk
    "A.C.I.L B4NKA ISLEMI!",                 # Karışık Manipülasyon
]
manipulation_results = []
for text in manipulated_texts:
    pred, prob = ai_predict_with_confidence(text)
    manipulation_results.append({'Metin': text, 'Tahmin': pred, 'Güven Skoru': prob})
pd.set_option('display.max_colwidth', None) # Metin sütununun tamamını göster
print(pd.DataFrame(manipulation_results))
pd.reset_option('display.max_colwidth')

print("----------------------------------------------------------------")

# 3.2. Semantic Ambiguity (Anlamsal Belirsizlik & False Positive Kontrolü)
print("\n3.2. Semantic Ambiguity (Anlamsal Belirsizlik & False Positive Kontrolü)")
context_cases = [
    "Az önce bir dolandırıcılık mesajı aldım, ne yapmalıyım?", # Bilgi alma, güvenli olmalı
    "Şifremi unuttuğum için giriş yapamıyorum.",              # Teknik sorun, güvenli olmalı
    "Hesap limitimi artırmak için şubeye mi gitmeliyim?",     # Rutin bankacılık, güvenli olmalı
    "Bankanızı arayan dolandırıcılara karşı korunma yolları." # Güvenli, "dolandırıcı" kelimesi geçiyor
]
context_results = []
for text in context_cases:
    pred, prob = ai_predict_with_confidence(text)
    context_results.append({'Metin': text, 'Tahmin': pred, 'Güven Skoru': prob})
pd.set_option('display.max_colwidth', None)
print(pd.DataFrame(context_results))
pd.reset_option('display.max_colwidth')

print("----------------------------------------------------------------")

# 3.3. Out-of-Distribution (Kapsam Dışı Veri / Anlamsız Girişler)
print("\n3.3. Out-of-Distribution (Kapsam Dışı Veri / Anlamsız Girişler)")
ood_cases = [
    "!!!CLICK NOW!!!",                   # Sadece yabancı dil
    "asdfghjkl123",                         # Anlamsız karakterler
    "SELECT * FROM users WHERE id=1",       # Potansiyel SQL Injection denemesi
    "......................."               # Sadece noktalama işaretleri
]
ood_results = []
for text in ood_cases:
    pred, prob = ai_predict_with_confidence(text)
    ood_results.append({'Metin': text, 'Tahmin': pred, 'Güven Skoru': prob})
pd.set_option('display.max_colwidth', None)
print(pd.DataFrame(ood_results))
pd.reset_option('display.max_colwidth')

print("----------------------------------------------------------------")

# 3.4. Overfitting vs. Professional Language (Maskelenmiş Dolandırıcılık)
print("\n3.4. Overfitting vs. Professional Language (Maskelenmiş Dolandırıcılık)")
stealth_cases = [
    "Değerli müşterimiz, 6698 sayılı KVKK kapsamında veri güncellemeniz gerekmektedir. Lütfen ekteki formu doldurunuz.", # Resmi dil
    "Bankamızın yeni güvenlik protokolü gereği tüm kullanıcıların sistem üzerinden onay vermesi zorunludur. Detaylar linkte.", # Resmi dil + yönlendirme
    "Yüksek düzeyli bir güvenlik açığı tespit edildi, hesabınızın dondurulmaması için son 15 dakika!" # Resmi ve acil
]
stealth_results = []
for text in stealth_cases:
    pred, prob = ai_predict_with_confidence(text)
    stealth_results.append({'Metin': text, 'Tahmin': pred, 'Güven Skoru': prob})
pd.set_option('display.max_colwidth', None)
print(pd.DataFrame(stealth_results))
pd.reset_option('display.max_colwidth')

In [ ]:
print("\n--- 4. PERFORMANS GÖRSELLEŞTİRMELERİ VE METRİKLER ---")

y_pred_train = model.predict(X_train_vec)
y_pred_test = model.predict(X_test_vec)

print("\n--- Eğitim Seti Performansı ---")
print(classification_report(y_train, y_pred_train, target_names=['Güvenli', 'Fraud']))

print("\n--- Test Seti Performansı ---")
print(classification_report(y_test, y_pred_test, target_names=['Güvenli', 'Fraud']))

# Karmaşıklık Matrisi (Confusion Matrix) - Test Seti İçin
cm_test = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tahmin: Güvenli', 'Tahmin: Fraud'],
            yticklabels=['Gerçek: Güvenli', 'Gerçek: Fraud'])
plt.ylabel('Gerçek Değer')
plt.xlabel('Model Tahmini')
plt.title('Model Hata Analizi (Confusion Matrix - Test Seti)')
plt.show()

# Performans Metrikleri Bar Grafiği (Test Seti İçin)
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_test),
    'Precision': precision_score(y_test, y_pred_test),
    'Recall': recall_score(y_test, y_pred_test),
    'F1-Score': f1_score(y_test, y_pred_test)
}
metric_names = list(metrics.keys())
metric_values = list(metrics.values())

plt.figure(figsize=(8, 5))
sns.barplot(x=metric_names, y=metric_values, hue=metric_names,palette='plasma',legend=False)
plt.title('Temel Performans Metrikleri (Test Seti)')
plt.ylabel('Değer')
plt.ylim(0, 1.1)
for index, value in enumerate(metric_values):
    plt.text(index, value + 0.02, f"{value:.2f}", ha='center')
plt.show()